# Error analysis

Goal: at least **100 errors** put into categories by hand, with numbers, examples and what to do about each category.

Steps:
1. `python scripts/evaluate.py errors` wrote `results/errors_to_annotate.csv` (one line per wrong prediction on the test set).
2. I open the CSV in a spreadsheet, fill the `category` column (one category per error) and `note`, and save it as `results/errors_annotated.csv`.
3. One week later I annotate 25 of them again in `results/errors_reannotated.csv` (columns `claim_id`, `category`) without looking at the first pass.
4. This notebook computes the tables and figures of the report.

If the SciFact test set gives fewer than 100 errors, I add errors from Climate-FEVER (last section).

## Categories

| category | when to use it |
| --- | --- |
| `evidence_not_retrieved` | the gold abstract never reached the verifier (column `stage` = retrieval_miss) |
| `wrong_sentences` | the right abstract was there but the 3 selected sentences miss the key one |
| `numeric` | the claim needs comparing numbers, units or percentages |
| `negation_quantifier` | not / no / all / only / at least ... changes the meaning |
| `multi_evidence` | several sentences or abstracts must be combined |
| `refuted_vs_nei` | evidence is close to the claim but does not really contradict it (or the opposite) |
| `domain_vocabulary` | synonyms or abbreviations of the field that the model does not link |
| `gold_label_questionable` | I think the dataset label is wrong or arguable |
| `other` | none of the above (explain in `note`) |

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from claimguard.metrics import cohen_kappa

RES = Path("../results")
auto = pd.read_csv(RES / "errors_to_annotate.csv")
print(len(auto), "errors exported")
auto.head()

## 1. What the automatic tags already say (before any manual work)

In [ ]:
print(auto["stage"].value_counts(), "\n")
print(pd.crosstab(auto["gold"], auto["pred"]), "\n")
print("errors with a number in the claim:", auto["has_number"].mean().round(3))
print("errors with a negation in the claim:", auto["has_negation"].mean().round(3))
# high-confidence errors are the dangerous ones: abstention cannot catch them
print(auto["confidence"].describe())
auto.sort_values("confidence", ascending=False).head(10)[["claim", "gold", "pred", "confidence", "stage"]]

## 2. Manual categories

In [ ]:
ann = pd.read_csv(RES / "errors_annotated.csv")
ann = ann[ann["category"].notna() & (ann["category"] != "")]
print(len(ann), "errors annotated (need at least 100)")
counts = ann["category"].value_counts()
pd.DataFrame({"n": counts, "share": (counts / len(ann)).round(3)})

In [ ]:
# category x stage, category x gold label, and mean confidence per category
display(pd.crosstab(ann["category"], ann["stage"]))
display(pd.crosstab(ann["category"], ann["gold"]))
display(ann.groupby("category")["confidence"].agg(["mean", "count"]).round(3).sort_values("mean", ascending=False))

In [ ]:
# two examples per category for the qualitative part of the report
for cat, group in ann.groupby("category"):
    print("=" * 8, cat, f"({len(group)})")
    for _, row in group.head(2).iterrows():
        print("claim   :", row["claim"])
        print("gold/pred:", row["gold"], "/", row["pred"], f"(conf {row['confidence']})")
        print("evidence:", str(row["top_evidence"])[:300])
        print("note    :", row["note"], "\n")

## 3. Am I consistent? (second annotation pass)

In [ ]:
again = pd.read_csv(RES / "errors_reannotated.csv")
both = ann.merge(again, on="claim_id", suffixes=("_first", "_second"))
kappa = cohen_kappa(both["category_first"].tolist(), both["category_second"].tolist())
print(f"{len(both)} errors annotated twice, agreement {np.mean(both['category_first'] == both['category_second']):.2f}, Cohen's kappa {kappa:.2f}")
both[both["category_first"] != both["category_second"]][["claim", "category_first", "category_second"]]

## 4. Figure for the report

In [ ]:
!cd .. && python scripts/make_figures.py

## 5. If there are fewer than 100 errors on SciFact

Climate-FEVER has 1,381 usable claims, so the verifier makes many more errors there.
The cell below exports a random sample of them in the same format (stage is always `verification` because the evidence is given).

In [ ]:
import random
from claimguard.data import load_climate_fever
from claimguard.pipeline import build_pipeline
from claimguard.utils import load_config
import os; os.chdir("..")
pipe = build_pipeline(load_config("configs/default.yaml"), with_corpus=False); pipe.audit = None
rows = []
for ex in load_climate_fever("data/raw/climate_fever/climate-fever.jsonl"):
    r = pipe.verify_with_evidence(ex.text, ex.evidence_texts)
    if r.label != ex.label:
        best = max(r.evidence, key=lambda e: max(e.probs[:2]))
        rows.append({"claim_id": "cf-" + ex.claim_id, "claim": ex.text, "gold": ex.label, "pred": r.label,
                     "confidence": round(r.confidence, 3), "stage": "verification", "top_evidence": best.text, "category": "", "note": ""})
random.Random(13).shuffle(rows)
need = max(0, 110 - len(auto))
pd.DataFrame(rows[:need]).to_csv("results/errors_to_annotate_climate.csv", index=False)
print(len(rows), "errors on Climate-FEVER,", need, "exported")
os.chdir("notebooks")